In [ ]:
pip install yfinance pandas numpy streamlit plotly

**ANNUAL VOLATILITY**

In [ ]:
#!/usr/bin/env python3
"""
One-period binomial option pricing using Yahoo Finance data.

Features:
- Pulls stock and option data from Yahoo Finance (via yfinance).
- Uses a live-ish spot price from Yahoo when available (otherwise falls back to close).
- Computes realized volatility from historical returns.
- Builds a one-period binomial tree (u, d).
- Prices a European call/put.
- Computes replicating portfolio (Delta, B).
- Computes portfolio value for a chosen number of option contracts.
- Supports periodic refresh for near real-time monitoring.
- Produces a basic investment decision using model vs market option price.
"""

from __future__ import annotations

import argparse
from dataclasses import dataclass
from datetime import datetime, timezone
import os
import sys
import time
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import yfinance as yf


@dataclass
class BinomialResult:
    spot: float
    strike: float
    maturity_years: float
    volatility: float
    r: float
    u: float
    d: float
    su: float
    sd: float
    payoff_up: float
    payoff_down: float
    delta: float
    bond: float
    risk_neutral_p: float
    model_price: float


def realized_volatility(close_prices: pd.Series, window: int = 60) -> float:
    """Annualized realized volatility using log returns."""
    log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
    if log_returns.empty:
        raise ValueError("Not enough history to compute returns.")
    if window > 1:
        log_returns = log_returns.tail(window)
    sigma_daily = float(log_returns.std(ddof=1))
    sigma_annual = sigma_daily * np.sqrt(252.0)
    if sigma_annual <= 0:
        raise ValueError("Computed volatility is non-positive.")
    return sigma_annual


def choose_expiry(ticker: yf.Ticker, user_expiry: Optional[str]) -> str:
    expiries = list(ticker.options)
    if not expiries:
        raise ValueError("No option expiries found for this ticker.")
    if user_expiry is None:
        return expiries[0]
    if user_expiry not in expiries:
        raise ValueError(
            f"Requested expiry {user_expiry} not available. "
            f"Available expiries start with: {expiries[:5]}"
        )
    return user_expiry


def choose_strike(
    chain_df: pd.DataFrame,
    spot: float,
    user_strike: Optional[float],
) -> float:
    strikes = chain_df["strike"].dropna().astype(float).values
    if strikes.size == 0:
        raise ValueError("No strikes available in option chain.")
    if user_strike is None:
        return float(strikes[np.argmin(np.abs(strikes - spot))])
    return float(strikes[np.argmin(np.abs(strikes - user_strike))])


def extract_market_option_price(
    chain_df: pd.DataFrame,
    strike: float,
) -> float:
    row = chain_df.loc[(chain_df["strike"] - strike).abs().idxmin()]
    bid = float(row.get("bid", 0.0) or 0.0)
    ask = float(row.get("ask", 0.0) or 0.0)
    last = float(row.get("lastPrice", np.nan))
    if bid > 0 and ask > 0:
        return (bid + ask) / 2.0
    if np.isfinite(last) and last > 0:
        return last
    raise ValueError("Could not infer market option price from bid/ask/last.")


def get_live_spot_price(ticker: yf.Ticker, fallback_close: float) -> float:
    """
    Try to get a live/delayed market price from Yahoo fast_info.
    If unavailable, return the latest historical close.
    """
    try:
        info = ticker.fast_info
        if hasattr(info, "get"):
            for key in ("lastPrice", "last_price", "regularMarketPrice", "regular_market_price"):
                value = info.get(key, None)
                if value is not None and float(value) > 0:
                    return float(value)
    except Exception:
        pass
    return float(fallback_close)


def one_period_binomial(
    spot: float,
    strike: float,
    r: float,
    maturity_years: float,
    sigma: float,
    option_type: str,
) -> BinomialResult:
    dt = maturity_years
    u = float(np.exp(sigma * np.sqrt(dt)))
    d = float(np.exp(-sigma * np.sqrt(dt)))
    growth = float(np.exp(r * dt))

    if not (d < growth < u):
        raise ValueError(
            "No-arbitrage condition failed: need d < exp(rT) < u. "
            "Try a different maturity, rate, or volatility window."
        )

    su = spot * u
    sd = spot * d

    if option_type.lower() == "call":
        payoff_up = max(su - strike, 0.0)
        payoff_down = max(sd - strike, 0.0)
    elif option_type.lower() == "put":
        payoff_up = max(strike - su, 0.0)
        payoff_down = max(strike - sd, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    delta = (payoff_up - payoff_down) / (su - sd)
    bond = np.exp(-r * dt) * (payoff_up - delta * su)

    p = (growth - d) / (u - d)
    model_price_rn = np.exp(-r * dt) * (p * payoff_up + (1.0 - p) * payoff_down)
    model_price_rep = delta * spot + bond

    if abs(model_price_rn - model_price_rep) > 1e-8:
        raise RuntimeError("Replicating and risk-neutral prices do not match.")

    return BinomialResult(
        spot=spot,
        strike=strike,
        maturity_years=dt,
        volatility=sigma,
        r=r,
        u=u,
        d=d,
        su=su,
        sd=sd,
        payoff_up=payoff_up,
        payoff_down=payoff_down,
        delta=delta,
        bond=bond,
        risk_neutral_p=p,
        model_price=model_price_rep,
    )


def investment_signal(
    model_price: float,
    market_price: float,
    threshold: float,
) -> str:
    relative_gap = (model_price - market_price) / market_price
    if relative_gap > threshold:
        return "BUY option (model > market: potentially undervalued)"
    if relative_gap < -threshold:
        return "SELL/WRITE option (model < market: potentially overvalued)"
    return "HOLD/NO TRADE (difference within threshold band)"


def render_live_plots(
    snapshots: List[Dict[str, Any]],
    plot_points: int,
    figure_state: Optional[Any] = None,
    save_plot_path: Optional[str] = None,
) -> Any:
    """
    Live visualization:
    - Left: recent stock price + one-step binomial move annotations.
    - Right: option/portfolio value comparison + key model parameters.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise ImportError(
            "matplotlib is required for --plot-live. Install with: pip install matplotlib"
        ) from exc

    if not snapshots:
        return figure_state

    rows = len(snapshots)
    if figure_state is None:
        plt.ion()
        figure_state = plt.figure(figsize=(16, 5 * rows))

    figure_state.clf()
    axes = figure_state.subplots(nrows=rows, ncols=2, squeeze=False)

    for i, snap in enumerate(snapshots):
        result = snap["result"]
        close_series = snap["close_series"].tail(plot_points)
        if close_series.empty:
            continue

        ax_price = axes[i][0]
        ax_value = axes[i][1]

        last_t = close_series.index[-1]
        days_to_expiry = max(1, int(round(result.maturity_years * 365.0)))
        next_t = last_t + pd.Timedelta(days=days_to_expiry)

        # Price chart with key option/binomial markers.
        ax_price.plot(close_series.index, close_series.values, lw=1.8, color="#1f77b4", label="Close")
        ax_price.scatter([last_t], [result.spot], s=30, color="black", zorder=5, label="S0")
        ax_price.axhline(snap["strike"], color="#ff7f0e", linestyle="--", lw=1.2, label="Strike K")
        ax_price.axhline(result.su, color="#2ca02c", linestyle=":", lw=1.2, label="Su")
        ax_price.axhline(result.sd, color="#d62728", linestyle=":", lw=1.2, label="Sd")
        ax_price.plot([last_t, next_t], [result.spot, result.su], color="#2ca02c", lw=1.5)
        ax_price.plot([last_t, next_t], [result.spot, result.sd], color="#d62728", lw=1.5)
        ax_price.scatter([next_t, next_t], [result.su, result.sd], c=["#2ca02c", "#d62728"], s=30, zorder=5)

        ax_price.annotate(f"S0={result.spot:.2f}", (last_t, result.spot), textcoords="offset points", xytext=(6, 6))
        ax_price.annotate(f"Su={result.su:.2f}", (next_t, result.su), textcoords="offset points", xytext=(6, 6))
        ax_price.annotate(f"Sd={result.sd:.2f}", (next_t, result.sd), textcoords="offset points", xytext=(6, -14))
        ax_price.annotate(f"K={snap['strike']:.2f}", (last_t, snap["strike"]), textcoords="offset points", xytext=(6, 6))

        ax_price.set_title(
            f"{snap['symbol']} Price + One-Period Binomial Tree (Expiry {snap['expiry']})",
            fontsize=11,
        )
        ax_price.set_xlabel("Time")
        ax_price.set_ylabel("Price")
        ax_price.grid(alpha=0.25)
        ax_price.legend(loc="best", fontsize=8)

        # Value chart for option and replicating portfolio totals.
        labels = ["Model total", "Market total", "Rep now", "Rep up", "Rep down"]
        values = [
            snap["model_total"],
            snap["market_total"],
            snap["replicate_value_now"],
            snap["replicate_value_up"],
            snap["replicate_value_down"],
        ]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#2ca02c", "#d62728"]

        bars = ax_value.bar(labels, values, color=colors, alpha=0.9)
        for bar, val in zip(bars, values):
            ax_value.text(
                bar.get_x() + bar.get_width() / 2.0,
                bar.get_height(),
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

        info_text = (
            f"u={result.u:.4f}, d={result.d:.4f}\n"
            f"sigma={result.volatility:.2%}, r={result.r:.2%}\n"
            f"Delta={result.delta:.4f}, B={result.bond:.4f}\n"
            f"Signal: {snap['signal']}"
        )
        ax_value.text(
            0.02,
            0.98,
            info_text,
            transform=ax_value.transAxes,
            va="top",
            ha="left",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8),
        )

        ax_value.set_title(
            f"{snap['symbol']} Option / Portfolio Valuation ({snap['contracts']} x {snap['contract_size']})",
            fontsize=11,
        )
        ax_value.set_ylabel("Value")
        ax_value.grid(axis="y", alpha=0.25)

    figure_state.tight_layout()
    if save_plot_path:
        figure_state.savefig(save_plot_path, dpi=140, bbox_inches="tight")
    figure_state.canvas.draw()
    figure_state.canvas.flush_events()
    plt.pause(0.001)
    return figure_state


def is_notebook_or_colab() -> bool:
    return (
        "google.colab" in sys.modules
        or "ipykernel" in sys.modules
        or bool(os.environ.get("JPY_PARENT_PID"))
        or bool(os.environ.get("COLAB_RELEASE_TAG"))
    )


def is_headless_environment() -> bool:
    if os.name == "nt":
        return False
    return not bool(os.environ.get("DISPLAY"))


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="One-period binomial option pricing with Yahoo Finance data."
    )
    parser.add_argument(
        "--tickers",
        type=str,
        default="AAPL,MSFT,TSLA,NVDA,RELIANCE,GOOGL,AMZN",
        help="Comma-separated tickers (default: AAPL,MSFT,TSLA).",
    )
    parser.add_argument(
        "--option-type",
        type=str,
        choices=["call", "put"],
        default="call",
        help="European option type (call or put).",
    )
    parser.add_argument(
        "--expiry",
        type=str,
        default=None,
        help="Expiry in YYYY-MM-DD. If omitted, nearest expiry is used.",
    )
    parser.add_argument(
        "--strike",
        type=float,
        default=None,
        help="Target strike. If omitted, nearest ATM strike is used.",
    )
    parser.add_argument(
        "--risk-free-rate",
        type=float,
        default=0.05,
        help="Annual continuously compounded risk-free rate (e.g., 0.05).",
    )
    parser.add_argument(
        "--history-period",
        type=str,
        default="1y",
        help="Yahoo history period for volatility estimation (default: 1y).",
    )
    parser.add_argument(
        "--vol-window",
        type=int,
        default=60,
        help="Number of recent trading days used for realized volatility.",
    )
    parser.add_argument(
        "--threshold",
        type=float,
        default=0.05,
        help="Decision threshold (default: 0.05 means 5%% mispricing band).",
    )
    parser.add_argument(
        "--contracts",
        type=int,
        default=1,
        help="Number of option contracts for portfolio valuation (default: 1).",
    )
    parser.add_argument(
        "--contract-size",
        type=int,
        default=100,
        help="Shares per option contract (default: 100 for US equity options).",
    )
    parser.add_argument(
        "--refresh-seconds",
        type=int,
        default=0,
        help="If > 0, reruns periodically every N seconds for live monitoring.",
    )
    parser.add_argument(
        "--plot-live",
        action="store_true",
        help="Enable live matplotlib charts with important model annotations.",
    )
    parser.add_argument(
        "--plot-points",
        type=int,
        default=120,
        help="Number of most recent close points to display in live plot.",
    )
    parser.add_argument(
        "--save-plot",
        type=str,
        default=None,
        help="Save the plot image to this path (useful in Colab/headless).",
    )
    # In Colab/Jupyter, kernels often inject: -f <kernel-connection-file>.
    # We ignore unknown args there so the script can run unchanged.
    if "google.colab" in sys.modules or "ipykernel" in sys.modules:
        args, _unknown = parser.parse_known_args()
        return args
    return parser.parse_args()


def run_for_ticker(
    symbol: str,
    args: argparse.Namespace,
) -> Dict[str, Any]:
    ticker = yf.Ticker(symbol)
    hist = ticker.history(period=args.history_period, auto_adjust=True)
    if hist.empty:
        raise ValueError(f"No historical price data for {symbol}.")

    last_close = float(hist["Close"].dropna().iloc[-1])
    spot = get_live_spot_price(ticker, fallback_close=last_close)
    sigma = realized_volatility(hist["Close"].dropna(), window=args.vol_window)

    expiry = choose_expiry(ticker, args.expiry)
    expiry_date = datetime.strptime(expiry, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)
    maturity_years = (expiry_date - now).total_seconds() / (365.0 * 24.0 * 3600.0)
    if maturity_years <= 0:
        raise ValueError(f"Expiry {expiry} is not in the future.")

    chain = ticker.option_chain(expiry)
    chain_df = chain.calls if args.option_type == "call" else chain.puts
    if chain_df.empty:
        raise ValueError(f"No {args.option_type} data found for {symbol} at {expiry}.")

    strike = choose_strike(chain_df, spot, args.strike)
    market_price = extract_market_option_price(chain_df, strike)

    result = one_period_binomial(
        spot=spot,
        strike=strike,
        r=args.risk_free_rate,
        maturity_years=maturity_years,
        sigma=sigma,
        option_type=args.option_type,
    )

    signal = investment_signal(
        model_price=result.model_price,
        market_price=market_price,
        threshold=args.threshold,
    )

    contract_multiplier = args.contracts * args.contract_size
    shares_units = result.delta * contract_multiplier
    cash_units_now = result.bond * contract_multiplier
    cash_units_at_expiry = cash_units_now * np.exp(result.r * result.maturity_years)

    replicate_value_now = result.model_price * contract_multiplier
    replicate_value_up = shares_units * result.su + cash_units_at_expiry
    replicate_value_down = shares_units * result.sd + cash_units_at_expiry
    model_total = result.model_price * contract_multiplier
    market_total = market_price * contract_multiplier

    print("\n" + "=" * 80)
    print(f"Ticker: {symbol}")
    print(f"Option: European {args.option_type.upper()} | Expiry: {expiry} | Strike: {strike:.2f}")
    print("-" * 80)
    print(f"Spot S0                 : {result.spot:.4f}")
    print(f"Realized volatility     : {result.volatility:.4%}")
    print(f"Maturity T (years)      : {result.maturity_years:.6f}")
    print(f"Risk-free rate r        : {result.r:.4%}")
    print(f"u, d                    : {result.u:.6f}, {result.d:.6f}")
    print(f"Su, Sd                  : {result.su:.4f}, {result.sd:.4f}")
    print(f"Payoff up/down          : {result.payoff_up:.4f}, {result.payoff_down:.4f}")
    print(f"Replicating Delta       : {result.delta:.6f}")
    print(f"Replicating Bond B      : {result.bond:.6f}")
    print(f"Risk-neutral probability: {result.risk_neutral_p:.6f}")
    print(f"Model option price      : {result.model_price:.4f}")
    print(f"Market option price     : {market_price:.4f}")
    print(f"Contracts x size        : {args.contracts} x {args.contract_size}")
    print(f"Replicating shares qty  : {shares_units:.4f}")
    print(f"Replicating cash today  : {cash_units_now:.4f}")
    print(f"Portfolio value now     : {replicate_value_now:.4f}")
    print(f"Portfolio value in up   : {replicate_value_up:.4f}")
    print(f"Portfolio value in down : {replicate_value_down:.4f}")
    print(f"Model portfolio value   : {model_total:.4f}")
    print(f"Market portfolio value  : {market_total:.4f}")
    print(f"Decision                : {signal}")
    print("=" * 80)

    return {
        "symbol": symbol,
        "expiry": expiry,
        "strike": strike,
        "result": result,
        "market_price": market_price,
        "signal": signal,
        "contracts": args.contracts,
        "contract_size": args.contract_size,
        "model_total": model_total,
        "market_total": market_total,
        "replicate_value_now": replicate_value_now,
        "replicate_value_up": replicate_value_up,
        "replicate_value_down": replicate_value_down,
        "close_series": hist["Close"].dropna(),
    }


def main() -> None:
    args = parse_args()
    tickers = [t.strip().upper() for t in args.tickers.split(",") if t.strip()]
    if not tickers:
        raise ValueError("Please provide at least one ticker.")

    print("Running one-period binomial pricing...")
    print(f"Tickers: {', '.join(tickers)}")
    print(f"Option type: {args.option_type}")
    if args.refresh_seconds > 0:
        print(f"Refresh interval: every {args.refresh_seconds} seconds")
    if args.plot_live:
        print(f"Live plotting: ON (last {args.plot_points} points)")

    auto_save_plot = None
    if args.plot_live and is_headless_environment() and not args.save_plot:
        auto_save_plot = "binomial_live_plot.png"
        print(f"No GUI display detected. Plot will be saved to: {auto_save_plot}")

    save_plot_path = args.save_plot or auto_save_plot
    figure_state = None
    try:
        while True:
            print(f"\nSnapshot UTC time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")
            snapshots: List[Dict[str, Any]] = []
            for symbol in tickers:
                try:
                    snapshot = run_for_ticker(symbol, args)
                    snapshots.append(snapshot)
                except Exception as exc:
                    print("\n" + "=" * 80)
                    print(f"Ticker: {symbol}")
                    print(f"Error: {exc}")
                    print("=" * 80)

            if args.plot_live:
                figure_state = render_live_plots(
                    snapshots=snapshots,
                    plot_points=args.plot_points,
                    figure_state=figure_state,
                    save_plot_path=save_plot_path,
                )

            if args.refresh_seconds <= 0:
                break
            time.sleep(args.refresh_seconds)
    except KeyboardInterrupt:
        print("\nStopped by user.")

    if args.plot_live and args.refresh_seconds <= 0:
        try:
            import matplotlib.pyplot as plt
            if save_plot_path:
                print(f"Saved plot: {save_plot_path}")
            # In desktop Python with GUI, keep the window open.
            if not is_notebook_or_colab() and not is_headless_environment():
                plt.show(block=True)
        except Exception:
            pass


if __name__ == "__main__":
    main()


Running one-period binomial pricing...
Tickers: AAPL, MSFT, TSLA, NVDA, RELIANCE, GOOGL, AMZN
Option type: call

Snapshot UTC time: 2026-04-19 08:01:07

Ticker: AAPL
Option: European CALL | Expiry: 2026-04-20 | Strike: 270.00
--------------------------------------------------------------------------------
Spot S0                 : 270.2300
Realized volatility     : 25.7659%
Maturity T (years)      : 0.001824
Risk-free rate r        : 5.0000%
u, d                    : 1.011066, 0.989055
Su, Sd                  : 273.2204, 267.2724
Payoff up/down          : 3.2204, 0.0000
Replicating Delta       : 0.541422
Replicating Bond B      : -144.693855
Risk-neutral probability: 0.501393
Model option price      : 1.6145
Market option price     : 1.8400
Contracts x size        : 1 x 100
Replicating shares qty  : 54.1422
Replicating cash today  : -14469.3855
Portfolio value now     : 161.4524
Portfolio value in up   : 322.0370
Portfolio value in down : 0.0000
Model portfolio value   : 161.4524
Marke

ERROR:yfinance:$RELIANCE: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")



Ticker: RELIANCE
Error: No historical price data for RELIANCE.

Ticker: GOOGL
Option: European CALL | Expiry: 2026-04-20 | Strike: 340.00
--------------------------------------------------------------------------------
Spot S0                 : 341.6800
Realized volatility     : 28.3136%
Maturity T (years)      : 0.001824
Risk-free rate r        : 5.0000%
u, d                    : 1.012166, 0.987980
Su, Sd                  : 345.8370, 337.5729
Payoff up/down          : 5.8370, 0.0000
Replicating Delta       : 0.706311
Replicating Bond B      : -238.409591
Risk-neutral probability: 0.500748
Model option price      : 2.9226
Market option price     : 3.4250
Contracts x size        : 1 x 100
Replicating shares qty  : 70.6311
Replicating cash today  : -23840.9591
Portfolio value now     : 292.2620
Portfolio value in up   : 583.7040
Portfolio value in down : 0.0000
Model portfolio value   : 292.2620
Market portfolio value  : 342.5000
Decision                : SELL/WRITE option (model < mark

**Considering volatility for last 60 minutes**

In [ ]:
#!/usr/bin/env python3
"""
One-period binomial option pricing using Yahoo Finance data.

Now supports 2 volatility modes:
1) daily: annualized volatility from daily close returns (original behavior)
2) intraday60: annualized volatility from LAST N minutes (default 60) of 1-minute data
"""

from __future__ import annotations

import argparse
from dataclasses import dataclass
from datetime import datetime, timezone
import os
import sys
import time
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf


@dataclass
class BinomialResult:
    spot: float
    strike: float
    maturity_years: float
    volatility: float
    r: float
    u: float
    d: float
    su: float
    sd: float
    payoff_up: float
    payoff_down: float
    delta: float
    bond: float
    risk_neutral_p: float
    model_price: float


def extract_close_series_from_download(df: pd.DataFrame, symbol: str) -> pd.Series:
    """Handle both normal and MultiIndex columns from yfinance.download."""
    if df.empty:
        return pd.Series(dtype=float)

    if isinstance(df.columns, pd.MultiIndex):
        # Most common: ('Close', symbol)
        if ("Close", symbol) in df.columns:
            s = df[("Close", symbol)]
        elif "Close" in df.columns.get_level_values(0):
            s = df.xs("Close", axis=1, level=0).iloc[:, 0]
        else:
            raise ValueError("Could not find Close column in intraday data.")
    else:
        if "Close" not in df.columns:
            raise ValueError("Could not find Close column in intraday data.")
        s = df["Close"]

    return pd.to_numeric(s, errors="coerce").dropna()


def realized_volatility_daily(close_prices: pd.Series, window: int = 60) -> float:
    """Annualized volatility from daily close log returns."""
    log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
    if log_returns.empty:
        raise ValueError("Not enough history to compute daily volatility.")
    if window > 1:
        log_returns = log_returns.tail(window)
    sigma_daily = float(log_returns.std(ddof=1))
    sigma_annual = sigma_daily * np.sqrt(252.0)
    if sigma_annual <= 0:
        raise ValueError("Computed daily volatility is non-positive.")
    return sigma_annual


def realized_volatility_last_minutes(
    symbol: str,
    interval: str = "1m",
    period: str = "5d",
    window_minutes: int = 60,
) -> Tuple[float, float, pd.Series]:
    """
    Returns:
    - sigma_annual: annualized vol from 1-min returns
    - sigma_1h: one-hour volatility (non-annualized)
    - close_tail: the close series used (last window_minutes points)
    """
    raw = yf.download(
        symbol,
        period=period,
        interval=interval,
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    close = extract_close_series_from_download(raw, symbol)
    close_tail = close.tail(window_minutes)

    if len(close_tail) < max(10, window_minutes // 3):
        raise ValueError(
            f"Not enough intraday bars for last {window_minutes} minutes. "
            f"Found {len(close_tail)} bars."
        )

    r_1m = np.log(close_tail / close_tail.shift(1)).dropna()
    if len(r_1m) < 2:
        raise ValueError("Not enough 1-minute returns for volatility.")

    sigma_1m = float(r_1m.std(ddof=1))
    sigma_1h = sigma_1m * np.sqrt(60.0)

    minutes_per_year = 252.0 * 6.5 * 60.0
    sigma_annual = sigma_1m * np.sqrt(minutes_per_year)

    if sigma_annual <= 0:
        raise ValueError("Computed intraday volatility is non-positive.")

    return float(sigma_annual), float(sigma_1h), close_tail


def choose_expiry(ticker: yf.Ticker, user_expiry: Optional[str]) -> str:
    expiries = list(ticker.options)
    if not expiries:
        raise ValueError("No option expiries found for this ticker.")
    if user_expiry is None:
        return expiries[0]
    if user_expiry not in expiries:
        raise ValueError(
            f"Requested expiry {user_expiry} not available. "
            f"Available expiries start with: {expiries[:5]}"
        )
    return user_expiry


def choose_strike(chain_df: pd.DataFrame, spot: float, user_strike: Optional[float]) -> float:
    strikes = chain_df["strike"].dropna().astype(float).values
    if strikes.size == 0:
        raise ValueError("No strikes available in option chain.")
    if user_strike is None:
        return float(strikes[np.argmin(np.abs(strikes - spot))])
    return float(strikes[np.argmin(np.abs(strikes - user_strike))])


def extract_market_option_price(chain_df: pd.DataFrame, strike: float) -> float:
    row = chain_df.loc[(chain_df["strike"] - strike).abs().idxmin()]
    bid = float(row.get("bid", 0.0) or 0.0)
    ask = float(row.get("ask", 0.0) or 0.0)
    last = float(row.get("lastPrice", np.nan))
    if bid > 0 and ask > 0:
        return (bid + ask) / 2.0
    if np.isfinite(last) and last > 0:
        return last
    raise ValueError("Could not infer market option price from bid/ask/last.")


def get_live_spot_price(ticker: yf.Ticker, fallback_close: float) -> float:
    try:
        info = ticker.fast_info
        if hasattr(info, "get"):
            for key in ("lastPrice", "last_price", "regularMarketPrice", "regular_market_price"):
                value = info.get(key, None)
                if value is not None and float(value) > 0:
                    return float(value)
    except Exception:
        pass
    return float(fallback_close)


def one_period_binomial(
    spot: float,
    strike: float,
    r: float,
    maturity_years: float,
    sigma_annual: float,
    option_type: str,
) -> BinomialResult:
    dt = maturity_years
    u = float(np.exp(sigma_annual * np.sqrt(dt)))
    d = float(np.exp(-sigma_annual * np.sqrt(dt)))
    growth = float(np.exp(r * dt))

    if not (d < growth < u):
        raise ValueError("No-arbitrage condition failed: need d < exp(rT) < u.")

    su = spot * u
    sd = spot * d

    if option_type.lower() == "call":
        payoff_up = max(su - strike, 0.0)
        payoff_down = max(sd - strike, 0.0)
    elif option_type.lower() == "put":
        payoff_up = max(strike - su, 0.0)
        payoff_down = max(strike - sd, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    delta = (payoff_up - payoff_down) / (su - sd)
    bond = np.exp(-r * dt) * (payoff_up - delta * su)

    p = (growth - d) / (u - d)
    model_price_rn = np.exp(-r * dt) * (p * payoff_up + (1.0 - p) * payoff_down)
    model_price_rep = delta * spot + bond

    if abs(model_price_rn - model_price_rep) > 1e-8:
        raise RuntimeError("Replicating and risk-neutral prices do not match.")

    return BinomialResult(
        spot=spot,
        strike=strike,
        maturity_years=dt,
        volatility=sigma_annual,
        r=r,
        u=u,
        d=d,
        su=su,
        sd=sd,
        payoff_up=payoff_up,
        payoff_down=payoff_down,
        delta=delta,
        bond=bond,
        risk_neutral_p=p,
        model_price=model_price_rep,
    )


def investment_signal(model_price: float, market_price: float, threshold: float) -> str:
    relative_gap = (model_price - market_price) / market_price
    if relative_gap > threshold:
        return "BUY option (model > market: potentially undervalued)"
    if relative_gap < -threshold:
        return "SELL/WRITE option (model < market: potentially overvalued)"
    return "HOLD/NO TRADE (difference within threshold band)"


def render_live_plots(
    snapshots: List[Dict[str, Any]],
    plot_points: int,
    figure_state: Optional[Any] = None,
    save_plot_path: Optional[str] = None,
) -> Any:
    try:
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise ImportError(
            "matplotlib is required for --plot-live. Install with: pip install matplotlib"
        ) from exc

    if not snapshots:
        return figure_state

    rows = len(snapshots)
    if figure_state is None:
        plt.ion()
        figure_state = plt.figure(figsize=(16, 5 * rows))

    figure_state.clf()
    axes = figure_state.subplots(nrows=rows, ncols=2, squeeze=False)

    for i, snap in enumerate(snapshots):
        result = snap["result"]
        close_series = snap["close_series"].tail(plot_points)
        if close_series.empty:
            continue

        ax_price = axes[i][0]
        ax_value = axes[i][1]

        last_t = close_series.index[-1]
        days_to_expiry = max(1, int(round(result.maturity_years * 365.0)))
        next_t = last_t + pd.Timedelta(days=days_to_expiry)

        ax_price.plot(close_series.index, close_series.values, lw=1.8, color="#1f77b4", label="Close")
        ax_price.scatter([last_t], [result.spot], s=30, color="black", zorder=5, label="S0")
        ax_price.axhline(snap["strike"], color="#ff7f0e", linestyle="--", lw=1.2, label="Strike K")
        ax_price.axhline(result.su, color="#2ca02c", linestyle=":", lw=1.2, label="Su")
        ax_price.axhline(result.sd, color="#d62728", linestyle=":", lw=1.2, label="Sd")
        ax_price.plot([last_t, next_t], [result.spot, result.su], color="#2ca02c", lw=1.5)
        ax_price.plot([last_t, next_t], [result.spot, result.sd], color="#d62728", lw=1.5)

        ax_price.set_title(
            f"{snap['symbol']} Price + One-Period Tree ({snap['expiry']})",
            fontsize=11,
        )
        ax_price.grid(alpha=0.25)
        ax_price.legend(loc="best", fontsize=8)

        labels = ["Model total", "Market total", "Rep now", "Rep up", "Rep down"]
        values = [
            snap["model_total"],
            snap["market_total"],
            snap["replicate_value_now"],
            snap["replicate_value_up"],
            snap["replicate_value_down"],
        ]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#2ca02c", "#d62728"]

        ax_value.bar(labels, values, color=colors, alpha=0.9)
        info_text = (
            f"vol_mode={snap['vol_mode']}\n"
            f"sigma_ann={result.volatility:.2%}\n"
            f"u={result.u:.4f}, d={result.d:.4f}\n"
            f"Delta={result.delta:.4f}, B={result.bond:.4f}\n"
            f"Signal: {snap['signal']}"
        )
        ax_value.text(
            0.02,
            0.98,
            info_text,
            transform=ax_value.transAxes,
            va="top",
            ha="left",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8),
        )
        ax_value.set_title(f"{snap['symbol']} Option / Portfolio Value", fontsize=11)
        ax_value.grid(axis="y", alpha=0.25)

    figure_state.tight_layout()
    if save_plot_path:
        figure_state.savefig(save_plot_path, dpi=140, bbox_inches="tight")
    figure_state.canvas.draw()
    figure_state.canvas.flush_events()
    plt.pause(0.001)
    return figure_state


def is_notebook_or_colab() -> bool:
    return (
        "google.colab" in sys.modules
        or "ipykernel" in sys.modules
        or bool(os.environ.get("JPY_PARENT_PID"))
        or bool(os.environ.get("COLAB_RELEASE_TAG"))
    )


def is_headless_environment() -> bool:
    if os.name == "nt":
        return False
    return not bool(os.environ.get("DISPLAY"))


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="One-period binomial pricing with Yahoo data.")

    parser.add_argument("--tickers", type=str, default="AAPL,MSFT,TSLA,NVDA,RELIANCE")
    parser.add_argument("--option-type", type=str, choices=["call", "put"], default="call")
    parser.add_argument("--expiry", type=str, default=None)
    parser.add_argument("--strike", type=float, default=None)
    parser.add_argument("--risk-free-rate", type=float, default=0.05)

    parser.add_argument("--history-period", type=str, default="1y")
    parser.add_argument("--vol-window", type=int, default=60)

    # NEW: volatility mode
    parser.add_argument(
        "--vol-mode",
        type=str,
        choices=["daily", "intraday60"],
        default="daily",
        help="daily: daily close returns | intraday60: last N minutes 1m returns.",
    )
    parser.add_argument("--intraday-interval", type=str, default="1m")
    parser.add_argument("--intraday-period", type=str, default="5d")
    parser.add_argument("--intraday-window-minutes", type=int, default=60)

    # Optional: force one-hour model step (instead of time-to-expiry)
    parser.add_argument(
        "--one-hour-step",
        action="store_true",
        help="Use T = 1 trading hour for binomial step.",
    )

    parser.add_argument("--threshold", type=float, default=0.05)
    parser.add_argument("--contracts", type=int, default=1)
    parser.add_argument("--contract-size", type=int, default=100)
    parser.add_argument("--refresh-seconds", type=int, default=0)

    parser.add_argument("--plot-live", action="store_true")
    parser.add_argument("--plot-points", type=int, default=120)
    parser.add_argument("--save-plot", type=str, default=None)

    if "google.colab" in sys.modules or "ipykernel" in sys.modules:
        args, _ = parser.parse_known_args()
        return args
    return parser.parse_args()


def run_for_ticker(symbol: str, args: argparse.Namespace) -> Dict[str, Any]:
    ticker = yf.Ticker(symbol)

    hist = ticker.history(period=args.history_period, auto_adjust=True)
    if hist.empty:
        raise ValueError(f"No historical data for {symbol}.")
    close_daily = hist["Close"].dropna()
    last_close = float(close_daily.iloc[-1])
    spot = get_live_spot_price(ticker, fallback_close=last_close)

    # Volatility selection
    sigma_1h = None
    vol_source = ""
    close_for_plot = close_daily
    if args.vol_mode == "intraday60":
        sigma_annual, sigma_1h, close_intraday = realized_volatility_last_minutes(
            symbol=symbol,
            interval=args.intraday_interval,
            period=args.intraday_period,
            window_minutes=args.intraday_window_minutes,
        )
        sigma = sigma_annual
        close_for_plot = close_intraday
        vol_source = (
            f"intraday {args.intraday_interval}, last {args.intraday_window_minutes} min "
            f"(period={args.intraday_period})"
        )
    else:
        sigma = realized_volatility_daily(close_daily, window=args.vol_window)
        vol_source = f"daily close returns, window={args.vol_window}"

    expiry = choose_expiry(ticker, args.expiry)
    expiry = '2026-05-15'
    expiry_date = datetime.strptime(expiry, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)
    t_to_expiry = (expiry_date - now).total_seconds() / (365.0 * 24.0 * 3600.0)
    if t_to_expiry <= 0:
        raise ValueError(f"Expiry {expiry} is not in the future.")

    # If requested, use one-hour step instead of expiry horizon
    if args.one_hour_step:
        maturity_years = 1.0 / (252.0 * 6.5)
    else:
        maturity_years = t_to_expiry

    chain = ticker.option_chain(expiry)
    chain_df = chain.calls if args.option_type == "call" else chain.puts
    if chain_df.empty:
        raise ValueError(f"No {args.option_type} data for {symbol} at {expiry}.")

    strike = choose_strike(chain_df, spot, args.strike)
    market_price = extract_market_option_price(chain_df, strike)

    result = one_period_binomial(
        spot=spot,
        strike=strike,
        r=args.risk_free_rate,
        maturity_years=maturity_years,
        sigma_annual=sigma,
        option_type=args.option_type,
    )

    signal = investment_signal(result.model_price, market_price, args.threshold)

    mult = args.contracts * args.contract_size
    shares_units = result.delta * mult
    cash_units_now = result.bond * mult
    cash_units_at_step = cash_units_now * np.exp(result.r * result.maturity_years)

    replicate_value_now = result.model_price * mult
    replicate_value_up = shares_units * result.su + cash_units_at_step
    replicate_value_down = shares_units * result.sd + cash_units_at_step
    model_total = result.model_price * mult
    market_total = market_price * mult

    print("\n" + "=" * 90)
    print(f"Ticker: {symbol}")
    print(f"Option: European {args.option_type.upper()} | Expiry used for chain: {expiry} | Strike: {strike:.2f}")
    print("-" * 90)
    print(f"Vol source              : {vol_source}")
    print(f"Sigma annual            : {result.volatility:.4%}")
    if sigma_1h is not None:
        print(f"Sigma over 1 hour       : {sigma_1h:.6f} (non-annualized)")
    print(f"Spot S0                 : {result.spot:.4f}")
    print(f"Maturity T (years)      : {result.maturity_years:.8f} "
          f"({'1-hour step' if args.one_hour_step else 'to expiry'})")
    print(f"Risk-free rate r        : {result.r:.4%}")
    print(f"u, d                    : {result.u:.6f}, {result.d:.6f}")
    print(f"Su, Sd                  : {result.su:.4f}, {result.sd:.4f}")
    print(f"Payoff up/down          : {result.payoff_up:.4f}, {result.payoff_down:.4f}")
    print(f"Replicating Delta       : {result.delta:.6f}")
    print(f"Replicating Bond B      : {result.bond:.6f}")
    print(f"Risk-neutral probability: {result.risk_neutral_p:.6f}")
    print(f"Model option price      : {result.model_price:.4f}")
    print(f"Market option price     : {market_price:.4f}")
    print(f"Contracts x size        : {args.contracts} x {args.contract_size}")
    print(f"Replicating shares qty  : {shares_units:.4f}")
    print(f"Replicating cash today  : {cash_units_now:.4f}")
    print(f"Portfolio value now     : {replicate_value_now:.4f}")
    print(f"Portfolio value in up   : {replicate_value_up:.4f}")
    print(f"Portfolio value in down : {replicate_value_down:.4f}")
    print(f"Model portfolio value   : {model_total:.4f}")
    print(f"Market portfolio value  : {market_total:.4f}")
    print(f"Decision                : {signal}")
    print("=" * 90)

    return {
        "symbol": symbol,
        "expiry": expiry,
        "strike": strike,
        "result": result,
        "market_price": market_price,
        "signal": signal,
        "contracts": args.contracts,
        "contract_size": args.contract_size,
        "model_total": model_total,
        "market_total": market_total,
        "replicate_value_now": replicate_value_now,
        "replicate_value_up": replicate_value_up,
        "replicate_value_down": replicate_value_down,
        "close_series": close_for_plot,
        "vol_mode": args.vol_mode,
    }


def main() -> None:
    args = parse_args()
    tickers = [t.strip().upper() for t in args.tickers.split(",") if t.strip()]
    if not tickers:
        raise ValueError("Please provide at least one ticker.")

    print("Running one-period binomial pricing...")
    print(f"Tickers: {', '.join(tickers)}")
    print(f"Option type: {args.option_type}")
    print(f"Vol mode: {args.vol_mode}")
    if args.refresh_seconds > 0:
        print(f"Refresh interval: every {args.refresh_seconds} seconds")
    if args.plot_live:
        print(f"Live plotting: ON (last {args.plot_points} points)")

    auto_save_plot = None
    if args.plot_live and is_headless_environment() and not args.save_plot:
        auto_save_plot = "binomial_live_plot.png"
        print(f"No GUI display detected. Plot will be saved to: {auto_save_plot}")

    save_plot_path = args.save_plot or auto_save_plot
    figure_state = None

    try:
        while True:
            print(f"\nSnapshot UTC: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")
            snapshots: List[Dict[str, Any]] = []
            for symbol in tickers:
                try:
                    snapshots.append(run_for_ticker(symbol, args))
                except Exception as exc:
                    print("\n" + "=" * 90)
                    print(f"Ticker: {symbol}")
                    print(f"Error: {exc}")
                    print("=" * 90)

            if args.plot_live:
                figure_state = render_live_plots(
                    snapshots=snapshots,
                    plot_points=args.plot_points,
                    figure_state=figure_state,
                    save_plot_path=save_plot_path,
                )

            if args.refresh_seconds <= 0:
                break
            time.sleep(args.refresh_seconds)

    except KeyboardInterrupt:
        print("\nStopped by user.")

    if args.plot_live and args.refresh_seconds <= 0:
        try:
            import matplotlib.pyplot as plt
            if save_plot_path:
                print(f"Saved plot: {save_plot_path}")
            if not is_notebook_or_colab() and not is_headless_environment():
                plt.show(block=True)
        except Exception:
            pass


if __name__ == "__main__":
    main()


Running one-period binomial pricing...
Tickers: AAPL, MSFT, TSLA, NVDA, RELIANCE
Option type: call
Vol mode: daily

Snapshot UTC: 2026-04-19 08:02:28

Ticker: AAPL
Option: European CALL | Expiry used for chain: 2026-05-15 | Strike: 270.00
------------------------------------------------------------------------------------------
Vol source              : daily close returns, window=60
Sigma annual            : 25.7659%
Spot S0                 : 270.2300
Maturity T (years)      : 0.07031491 (to expiry)
Risk-free rate r        : 5.0000%
u, d                    : 1.070712, 0.933958
Su, Sd                  : 289.3384, 252.3836
Payoff up/down          : 19.3384, 0.0000
Replicating Delta       : 0.523298
Replicating Bond B      : -131.608388
Risk-neutral probability: 0.508680
Model option price      : 9.8025
Market option price     : 8.7500
Contracts x size        : 1 x 100
Replicating shares qty  : 52.3298
Replicating cash today  : -13160.8388
Portfolio value now     : 980.2526
Portfolio val

ERROR:yfinance:$RELIANCE: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")



Ticker: RELIANCE
Error: No historical data for RELIANCE.
